# LightOnOCR

In [ ]:
import os
import requests

url = os.environ.get(
    "PREDICTOR_URL",
    "http://lightonocr-2-1b-predictor.model-serving.svc.cluster.local:8080/v1/chat/completions",
)
# "model" must match InferenceService metadata.name (vLLM --served-model-name={{.Name}})
# This model needs <image>\n at the *start* of the user text, then the instruction.
EXAMPLE_URL ="https://www.mattmahoney.net/ocr/numbers_gs150.jpg"
EXAMPLE_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"

user_prompt = os.environ.get("LIGHTON_OCR_PROMPT", "<image>\nExtract the items and total price from this receipt.")
payload = {
    "model": os.environ.get("INFERENCE_MODEL", "lightonocr-2-1b"),
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": os.environ.get("LIGHTON_OCR_IMAGE_URL", EXAMPLE_URL)},
                },
            ],
        }
    ],
    "max_tokens": int(os.environ.get("MAX_TOKENS", "1024")),
    "temperature": 0.0,
}
h = {"Content-Type": "application/json"}
tok = os.environ.get("INFERENCE_BEARER_TOKEN") or os.environ.get("OC_TOKEN")
if tok:
    h["Authorization"] = f"Bearer {tok}"
r = requests.post(url, json=payload, headers=h, timeout=300, verify=False)
if r.status_code == 200:
    print("✅ OCR Result:")
    print(r.json()["choices"][0]["message"]["content"])
else:
    print(f"❌ Error: {r.status_code}")
    print(r.text[:2000])